In [18]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ==========
# PATHS (Windows anpassen)
# ==========
BASELINE_ROOT = Path(r"C:\Dev\Bachelorarbeit\results\accounting\runs\cpcv_baseline")
FINAL_ROOT    = Path(r"C:\Dev\Bachelorarbeit\results\accounting\runs\cpcv_final")
OUT_DIR       = Path(r"C:\Dev\Bachelorarbeit\results\accounting\plots_cpcv")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ==========
# Load all dispersion_over_paths.csv from a root
# ==========
def load_dispersion(root: Path, experiment_name: str) -> pd.DataFrame:
    rows = []
    for csv_path in root.rglob("dispersion_over_paths.csv"):
        # .../<config_dir>/<run_id>/<SB3_Defaults|E2>/_report/dispersion_over_paths.csv
        config_dir = csv_path.parents[3].name
        run_id     = csv_path.parents[2].name
        report_run = csv_path.parents[1].name  # SB3_Defaults oder E2

        cfg_clean = config_dir.replace("_final", "")
        m = re.match(r"(?P<reward>log|icvar|icvar_dd)_ppo_(?P<state>S[01])", cfg_clean)
        if m:
            config_label = f"{m.group('reward')}-{m.group('state')}"
        else:
            config_label = cfg_clean

        df = pd.read_csv(csv_path)
        df = df.rename(columns={"run": "report_run"})
        df["experiment"]  = experiment_name
        df["config"]      = config_label
        df["config_dir"]  = config_dir
        df["run_id"]      = run_id
        df["report_group"] = report_run

        rows.append(df)

    if not rows:
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)

df_base = load_dispersion(BASELINE_ROOT, "baseline")
df_final = load_dispersion(FINAL_ROOT, "final")
df_all = pd.concat([df_base, df_final], ignore_index=True)

print(df_all.shape)
print(df_all[["experiment","config","metric"]].drop_duplicates().head(20))

print("OUT_DIR =", OUT_DIR)
print("BASELINE_ROOT exists:", BASELINE_ROOT.exists())
print("FINAL_ROOT exists:", FINAL_ROOT.exists())
print("baseline csv count:", len(list(BASELINE_ROOT.rglob("dispersion_over_paths.csv"))))
print("final csv count:", len(list(FINAL_ROOT.rglob("dispersion_over_paths.csv"))))



(132, 15)
   experiment       config            metric
0    baseline  icvar_dd-S0  total_cum_return
1    baseline  icvar_dd-S0       total_maxdd
2    baseline  icvar_dd-S0     ex_cum_return
3    baseline  icvar_dd-S0         ex_sharpe
4    baseline  icvar_dd-S0        ex_sortino
5    baseline  icvar_dd-S0        ex_cvar_95
6    baseline  icvar_dd-S0         ex_calmar
7    baseline  icvar_dd-S0     avg_cost_rate
8    baseline  icvar_dd-S0      avg_turnover
9    baseline  icvar_dd-S0            psr_ex
10   baseline  icvar_dd-S0            dsr_ex
11   baseline  icvar_dd-S1  total_cum_return
12   baseline  icvar_dd-S1       total_maxdd
13   baseline  icvar_dd-S1     ex_cum_return
14   baseline  icvar_dd-S1         ex_sharpe
15   baseline  icvar_dd-S1        ex_sortino
16   baseline  icvar_dd-S1        ex_cvar_95
17   baseline  icvar_dd-S1         ex_calmar
18   baseline  icvar_dd-S1     avg_cost_rate
19   baseline  icvar_dd-S1      avg_turnover
OUT_DIR = C:\Dev\Bachelorarbeit\results\accou

In [21]:
# Reihenfolge für schöne konsistente Plots
CONFIG_ORDER = ["log-S0","log-S1","icvar-S0","icvar-S1","icvar_dd-S0","icvar_dd-S1"]

METRIC_LABELS = {
    "ex_cum_return" : "Excess Cumulative Return",
    "ex_sharpe": "Excess Sharpe",
    "total_maxdd": "Max Drawdown (total)",
    "ex_cvar_95": "Excess CVaR 95%",
    "avg_turnover": "Average Turnover",
    "avg_cost_rate": "Average Cost Rate"
}

MAIN_METRICS = ["ex_cum_return", "ex_sharpe", "total_maxdd", "ex_cvar_95", "avg_turnover"]


def plot_metric(df: pd.DataFrame, experiment: str, metric: str):
    sub = df[(df["experiment"] == experiment) & (df["metric"] == metric)].copy()
    if sub.empty:
        return


    need = ["median","q25","q75","min","max"]
    sub = sub.dropna(subset=need)
    if sub.empty:
        return

    sub["config"] = pd.Categorical(sub["config"], categories=CONFIG_ORDER, ordered=True)
    sub = sub.sort_values("config")

    stats = []
    labels = []
    for _, r in sub.iterrows():
        stats.append({
            "med": r["median"],
            "q1": r["q25"],
            "q3": r["q75"],
            "whislo": r["min"],
            "whishi": r["max"],
            "mean": r["mean"] if "mean" in sub.columns and pd.notna(r["mean"]) else None,
            "fliers": []
        })
        labels.append(str(r["config"]))

    fig, ax = plt.subplots(figsize=(10, 4.8))

    meanprops   = dict(marker="o", markerfacecolor="tab:blue", markeredgecolor="tab:blue", markersize=5)
    medianprops = dict(color="tab:orange", linewidth=2.0)
    boxprops    = dict(linewidth=1.2, facecolor="lightgray")
    whiskerprops= dict(linewidth=1.0)
    capprops    = dict(linewidth=1.0)

    res = ax.bxp(
        stats,
        showfliers=False,
        showmeans=True,
        meanline=False,
        patch_artist=True,      # wichtig: Box füllen
        widths=0.6,
        meanprops=meanprops,
        medianprops=medianprops,
        boxprops=boxprops,
        whiskerprops=whiskerprops,
        capprops=capprops,
    )

    # optional: alpha für die Boxen
    for b in res["boxes"]:
        b.set_alpha(0.35)


    ax.set_xticklabels(labels, rotation=0, ha="center")
    ax.tick_params(axis="both", labelsize=11)
    ax.set_title(f"CPCV {experiment}: {METRIC_LABELS.get(metric, metric)}", fontsize=14)

    ax.grid(True, axis="y", alpha=0.2, linewidth=0.8)

    legend_handles = [
    Line2D([0],[0], color="tab:orange", lw=2.0, label="Median"),
    Line2D([0],[0], marker="o", color="tab:blue", lw=0, markersize=5, label="Mean"),
    Patch(facecolor="lightgray", edgecolor="black", alpha=0.35, label="IQR (Q1–Q3)"),
    Line2D([0],[0], color="black", lw=1.0, label="Whiskers (Min–Max)"),
]
    ax.set_ylabel(METRIC_LABELS.get(metric, metric))

    ax.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.18),
        ncol=4,
        frameon=False,
        fontsize=10
    )
    fig.tight_layout()
    fig.subplots_adjust(bottom=0.25)


    out_pdf = OUT_DIR / f"cpcv_{experiment}_box_{metric}.pdf"
    out_png = OUT_DIR / f"cpcv_{experiment}_box_{metric}.png"
    fig.savefig(out_pdf)
    fig.savefig(out_png, dpi=200)
    plt.close(fig)

for exp in ["baseline", "final"]:
    for m in MAIN_METRICS:
        plot_metric(df_all, exp, m)

print("done plots ->", OUT_DIR)
print("pdfs:", len(list(OUT_DIR.glob("*.pdf"))))
print("pngs:", len(list(OUT_DIR.glob("*.png"))))




done plots -> C:\Dev\Bachelorarbeit\results\accounting\plots_cpcv
pdfs: 10
pngs: 10


In [20]:
# Kompakte Tabelle: alle Konfis x alle KPIs mit Streuungsmaßen
cols = ["experiment","config","metric","median","iqr","std","min","max","q25","q75","mean"]
cols = [c for c in cols if c in df_all.columns]

tbl = df_all[cols].copy()
tbl = tbl.sort_values(["experiment","config","metric"])

# getrennt exportieren
tbl[tbl["experiment"]=="baseline"].to_csv(OUT_DIR / "cpcv_dispersion_baseline.csv", index=False)
tbl[tbl["experiment"]=="final"].to_csv(OUT_DIR / "cpcv_dispersion_final.csv", index=False)

# optional: Pivot nur IQR als Heatmap-Input / schnelle Übersicht
pivot_iqr_final = tbl[tbl["experiment"]=="final"].pivot(index="config", columns="metric", values="iqr")
pivot_iqr_final.to_csv(OUT_DIR / "cpcv_final_iqr_pivot.csv")

print("done tables")


done tables
